# 5. Evaluation and Analysis - Akkadian to English Translation 
## Aaron Dichoso & Luis Razon

This notebook details the steps performed for evaluating the trained models and evaluating on the test data used in the Deep Past Challenge for Translating Akkadian Text to English.
The competition can be accessed in this link: https://www.kaggle.com/competitions/deep-past-initiative-machine-translation/data

Run this notebook AFTER obtaining the models from "3. ModelingTraining" and "4. MBart"

The first step is to import the test data and run it through the same steps in cleaning, preprocessing, and modeling.

In [1]:
import pandas as pd
import re

test_df = pd.read_csv("dataset/testdata.csv")
test_df.sample(1)

,id,translation,transliteration
1,1,In the letter of the City (it is written): Fro...,i-na mup-pì-im aa a-lim(ki) ia-tù u„-mì-im a-n...


## Test Data Cleaning

In [2]:
import model.cleaning as cleaning

In [3]:
test_df["transliteration"] = cleaning.process_transliterations(test_df)
test_df["translation"] = cleaning.process_translations(test_df)
test_df.sample(1)

,id,translation,transliteration
1,1,In the letter of the City it is written: From ...,i-na mup-pì-im aa a-lim(ki) ia-tù u-mì-im a-ni...


## Test Data Preprocessing - FOR MBart

Install and import dependencies for the MBart model. The MBart tokenizer handles its own preprocessing, so we use the cleaned 'test_df' directly, so we do not tokenize the dataset yet at this step.

In [4]:
from sacrebleu.metrics import BLEU, CHRF
import torch
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast
import numpy as np

#Get device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

c:\Users\Aaron\.conda\envs\thesis\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


### Configuration
Set the checkpoint path and generation hyperparameters below.

In [ ]:
#NOTE: AI was used to help understand the flow in setting up MBart Model for training and testing and for general debugging. 
#The following code, however, was written and verified by the authors. 

CHECKPOINT_PATH = "checkpoints/mbart/E5"

#MBart hyperparameters (must match training in 4. MBart.ipynb)
MAX_LEN        = 748
NUM_BEAMS      = 4
LENGTH_PENALTY = 0.6
BATCH_SIZE     = 1
SHOW_EXAMPLES  = 5
SAVE_RESULTS   = "submission.csv"
NUM_SAMPLES    = None

#Language tokens (must match training in 4. MBart.ipynb)
AKKADIAN_LANG_TOKEN = "akk_XX"
TARGET_LANG_TOKEN   = "en_XX"

### Load Model & Tokenizer

In [6]:
#Load MBart Tokenizer
mbart_tokenizer = MBart50TokenizerFast.from_pretrained(
    CHECKPOINT_PATH,
    src_lang="en_XX",
)

#Set custom source tokenizer (akk_XX from 4. MBart.ipynb)
akk_id = mbart_tokenizer.convert_tokens_to_ids(AKKADIAN_LANG_TOKEN)
mbart_tokenizer.lang_code_to_id[AKKADIAN_LANG_TOKEN] = akk_id

#Set source and target languages
mbart_tokenizer.src_lang = AKKADIAN_LANG_TOKEN
mbart_tokenizer.tgt_lang = TARGET_LANG_TOKEN

#Testing Mode
mbart_model = MBartForConditionalGeneration.from_pretrained(CHECKPOINT_PATH).to(device)
mbart_model.eval()

print(f"Vocab Size: {len(mbart_tokenizer)}")
print(f"Device: {device}")

Loading weights: 100%|██████████| 516/516 [00:01<00:00, 325.63it/s]


Vocab Size: 250162
Device: cuda


### Helper Functions

For metrics, we use the geometric mean between BLEU and chrF++, as specified by the competition evaluation rules.

In [7]:
def mbart_translate(texts, references, max_len=MAX_LEN, num_beams=NUM_BEAMS, teacher_forcing=True,
                    length_penalty=LENGTH_PENALTY, batch_size=BATCH_SIZE):
    #Translate a list of Akkadian transliteration strings to English.
    all_translations = []

    #Predict strings in batches
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = mbart_tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=max_len,
        ).to(device)

        #Test Mode, forward pass.
        with torch.no_grad():
            if teacher_forcing:
                # Tokenize ground-truth targets (English) as decoder input
                batch_ref = references[i : i + batch_size]
                mbart_tokenizer.src_lang = TARGET_LANG_TOKEN
                tgt_enc = mbart_tokenizer(
                    batch_ref,
                    return_tensors="pt",
                    padding=True,
                    truncation=True,
                    max_length=max_len,
                ).to(device)
                mbart_tokenizer.src_lang = AKKADIAN_LANG_TOKEN  # restore

                # Forward pass with ground-truth decoder input ids
                outputs = mbart_model(
                    input_ids=inputs["input_ids"],
                    attention_mask=inputs["attention_mask"],
                    decoder_input_ids=tgt_enc["input_ids"],
                )

                # Recover token ids from logits via argmax, then decode
                pred_ids = outputs.logits.argmax(-1)
                decoded = mbart_tokenizer.batch_decode(pred_ids, skip_special_tokens=True)
            else:
                # Standard autoregressive beam search
                generated = mbart_model.generate(
                    **inputs,
                    forced_bos_token_id=mbart_tokenizer.lang_code_to_id[TARGET_LANG_TOKEN],
                    max_length=max_len,
                    num_beams=num_beams,
                    length_penalty=length_penalty,
                    early_stopping=True,
                )
                decoded = mbart_tokenizer.batch_decode(generated, skip_special_tokens=True)
        all_translations.extend(decoded)
    return all_translations

def compute_mbart_metrics(hypotheses, references):
    refs = [references]   # sacrebleu expects a list of reference lists
    bleu_score = BLEU(effective_order=True).corpus_score(hypotheses, refs)
    chrf_score = CHRF().corpus_score(hypotheses, refs)
    geomean    = np.sqrt((bleu_score.score / 100) * (chrf_score.score / 100)) * 100
    return {
        "BLEU" : round(bleu_score.score, 4),
        "ChrF" : round(chrf_score.score, 4),
        "Geomean": round(geomean, 4)
    }

### Prepare Test Data for MBart
MBart uses its own built-in tokenizer, so we pull directly from the cleaned `test_df`.

In [8]:
mbart_test_df = test_df.copy()

if NUM_SAMPLES:
    mbart_test_df = mbart_test_df.sample(n=min(NUM_SAMPLES, len(mbart_test_df)), random_state=42).reset_index(drop=True)
    print(f"Evaluating on {len(mbart_test_df)} randomly sampled rows.")
else:
    print(f"Evaluating on all {len(mbart_test_df)} rows.")

mbart_sources = mbart_test_df["transliteration"].tolist()
mbart_references = mbart_test_df["translation"].tolist()

print(f"\nSample Akkadian : {mbart_sources[0][:100]}...")
print(f"Sample English  : {mbart_references[0][:100]}...")

Evaluating on all 4 rows.

Sample Akkadian : um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il<big_gap> da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim ...
Sample English  : Thus Kanesh, say to the payers, our messenger, every single colony, and the trading stations: A lett...


### MBart Testing (w/ Teacher Forcing)

For each network, we test its performance with and without teacher forcing. Adding teacher forcing shows similar performances during training and validation as these were implemented during those phases, so they serve as more of a sanity check that the model trained well.

For the actual submission, we predict the tokens without teacher forcing, as the english translations are not provided in the Kaggle submission.

Due to time limitations and hardware constraints, we were only able to train MBart with 5 epochs, so the current results may still be improved with longer training times.

In [9]:
mbart_predictions = mbart_translate(mbart_sources, mbart_references, teacher_forcing=True)

In [10]:
n_show = min(SHOW_EXAMPLES, len(mbart_sources))

print("Sample MBart Translations")
for idx in range(n_show):
    src  = mbart_sources[idx]
    ref  = mbart_references[idx]
    pred = mbart_predictions[idx]
    print(f"\n[{idx+1}] SOURCE: {src[:120]}{'...' if len(src)>120 else ''}")
    print(f"REFERENCE: {ref[:120]}{'...' if len(ref)>120 else ''}")
    print(f"PREDICTED: {pred[:120]}{'...' if len(pred)>120 else ''}")

Sample MBart Translations

[1] SOURCE: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il<big_gap> da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi-ma mup-pu-um a...
REFERENCE: Thus Kanesh, say to the payers, our messenger, every single colony, and the trading stations: A letter of the City has a...
PREDICTED: Fromhu senan esh sa say to A <er s, theur me ssen ger s, en y in gle y, an d the sra in g sta nd: The er from t Cit y ha...

[2] SOURCE: i-na mup-pì-im aa a-lim(ki) ia-tù u-mì-im a-nim ma-ma-an KÙ.AN i-aa-ú-mu-ni i-na né-mì-lim da-aùr ú-lá e-WA ia-ra-tí-au ...
REFERENCE: In the letter of the City it is written: From this day on, whoever buys meteoric iron, the City of Assur is not part of ...
PREDICTED: W the er of the Cit y an is im tten  ' tha s ma     er , s  foran ?ri c me on? w Kan y  thesur wis  la t.  it . n.  Kra ...

[3] SOURCE: ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in lu té-ra-at É.GAL-lim ú-kà-lim lu na-aí-m...
REFERENCE: As soon as y

The predictions show that the model has not yet fully understood the relationships between each tokens and is only starting to do so.

Sample Predictions that show this:
1. "Cit y an is im tten" vs "City it is written"
2. "her to an to the pal ace" vs "either sold it to a palace"
3. "theur me ssen ger s, en y in gle y" vs "our messenger, every single colony"

This supports our initial beliefs that further training can improve MBart.

In [11]:
mbart_metrics = compute_mbart_metrics(mbart_predictions, mbart_references)
print("MBart Metrics (WITH TEACHER FORCING)")

for name, value in mbart_metrics.items():
    print(f"{name}: {value}")

MBart Metrics (WITH TEACHER FORCING)
BLEU: 1.5151
ChrF: 34.53
Geomean: 7.233


MBart Metrics (WITH TEACHER FORCING):
1. BLEU: 1.5151
2. ChrF: 34.53
3. Geomean: 7.233

### MBart Testing (w/out Teacher Forcing)

In [12]:
mbart_predictions = mbart_translate(mbart_sources, mbart_references, teacher_forcing=False)

In [13]:
n_show = min(SHOW_EXAMPLES, len(mbart_sources))

print("Sample MBart Translations")
for idx in range(n_show):
    src  = mbart_sources[idx]
    ref  = mbart_references[idx]
    pred = mbart_predictions[idx]
    print(f"\n[{idx+1}] SOURCE: {src[:120]}{'...' if len(src)>120 else ''}")
    print(f"REFERENCE: {ref[:120]}{'...' if len(ref)>120 else ''}")
    print(f"PREDICTED: {pred[:120]}{'...' if len(pred)>120 else ''}")

Sample MBart Translations

[1] SOURCE: um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il<big_gap> da-tim aí-ip-ri-ni kà-ar kà-ar-ma ú wa-bar-ra-tim qí-bi-ma mup-pu-um a...
REFERENCE: Thus Kanesh, say to the payers, our messenger, every single colony, and the trading stations: A letter of the City has a...
PREDICTED: From the Kan esh colony to <bi g_ga p>, the <bi g_ga p>, the <bi g_ga p>, the <bi g_ga p>, the <bi g_ga p>, the <bi g_ga...

[2] SOURCE: i-na mup-pì-im aa a-lim(ki) ia-tù u-mì-im a-nim ma-ma-an KÙ.AN i-aa-ú-mu-ni i-na né-mì-lim da-aùr ú-lá e-WA ia-ra-tí-au ...
REFERENCE: In the letter of the City it is written: From this day on, whoever buys meteoric iron, the City of Assur is not part of ...
PREDICTED: <bi g_ga p> in the nam e of the Cit y <bi g_ga p> an al l thi s <bi g_ga p> a lot of gold <bi g_ga p> my me ssen ger <bi...

[3] SOURCE: ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na aí-mì-im a-na É.GAL-lim i-dí-in lu té-ra-at É.GAL-lim ú-kà-lim lu na-aí-m...
REFERENCE: As soon as y

In [14]:
mbart_metrics = compute_mbart_metrics(mbart_predictions, mbart_references)
print("MBart Metrics (NO TEACHER FORCING)")

for name, value in mbart_metrics.items():
    print(f"{name}: {value}")

MBart Metrics (NO TEACHER FORCING)
BLEU: 1.0609
ChrF: 23.3266
Geomean: 4.9747


MBart Metrics (NO TEACHER FORCING):
1. BLEU: 1.0609
2. ChrF: 23.3266
3. Geomean: 4.9747

Comparing our MBart predictions with and without teacher forcing shows interesting results. First, when teacher forcing is not present, the model usually resorts to using the <big_gap> token as a fallback. Additionally, places and proper nouns stand out as being easy tokens to predict (Kanesh, City, palace).

In [15]:
mbart_test_df["predicted_translation"] = mbart_predictions
mbart_test_df.to_csv(SAVE_RESULTS, index=False)
print(f"Results saved to: {SAVE_RESULTS}")
mbart_test_df[["transliteration", "translation", "predicted_translation"]].head()

Results saved to: submission.csv


,transliteration,translation,predicted_translation
0,um-ma kà-ru-um kà-ni-ia-ma a-na aa-qí-il<big_g...,"Thus Kanesh, say to the payers, our messenger,...","From the Kan esh colony to <bi g_ga p>, the <b..."
1,i-na mup-pì-im aa a-lim(ki) ia-tù u-mì-im a-ni...,In the letter of the City it is written: From ...,<bi g_ga p> in the nam e of the Cit y <bi g_ga...
2,ki-ma mup-pì-ni ta-áa-me-a-ni a-ma-kam lu a-na...,"As soon as you have heard our letter, whoever ...",<bi g_ga p> as soon as you agree wit h our agr...
3,me-e-er mup-pì-ni a-na kà-ar kà-ar-ma ú wa-bar...,Send a copy of this letter of ours to every si...,<bi g_ga p> my da ughter <bi g_ga p> I ha ve w...


## Test Data Modeling

This section would then focus on the performance of the transformer and lstm models that use the given tokens for prediction.

### Test Data Tokenization

In [16]:
import model.tokenize as tokenize

In [17]:
tokenized_df = tokenize.tokenize(test_df)
tokenized_df.sample(1)

Model loaded from processed/akk2eng.json


,id,translation,transliteration
2,2,"[<sos>, As_, soon_, as_, you_, have_, heard_, ...","[<sos>, ki, -, ma_, mu, p, -, p, ì, -, ni_, ta..."


In [18]:
import torch
import model.dataset as dataset
import model.bulk_tester as btester
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

import model.decoder as decoder
import model.test as test

Model loaded from processed/akk2eng.json


In [19]:
test_loader = dataset.get_test_loader(tokenized_df)

Test samples: 4


### Basic Transformer Testing

In [20]:
import model.transformer as transformer

#Model epochs with good performance according to geomean
model = transformer.load_transformer_checkpoint("checkpoints/seq2seq_epoch114_vAcc0.3087_vLoss4.3496.pth", device)

Loaded epoch 114 | Val Acc: 0.3087 | Val Loss: 4.3496


#### Transformer w/out Teacher Forcing

In [21]:
test.test_model(model, test_loader, device, 4, "t", "raw", False)

Testing:   0%|          | 0/4 [00:00<?, ?it/s]c:\Users\Aaron\.conda\envs\thesis\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(
                                                      

METRICS
Test Loss     : 8.6888
Test Accuracy : 0.0319
BLEU          : 2.5053
chrF++        : 18.0868
Geometric Mean: 6.7315
PRED : From the attorney to the City and the attorney and the City and the City assembly and the City assembly of the City assembly of the City and 
REF  : Thus Kanesh, say to the payers, our messenger, every single colony, and the trading stations: A letter of the City has arrived. 
PRED : To Ali-ahum from Aššur-malik and Aššur-malik and Aššur-malik and Aššur-malik owes the week of the week of the week of the City assembly of the week of the week of the 
REF  : In the letter of the City it is written: From this day on, whoever buys meteoric iron, the City of Assur is not part of the profit made, the tithe on it Kanesh will collect. 
PRED : From the attorney and Ennam-Aššur to Ennam-Aššur: As to the silver that you have not be not obstruct me a tablet concerning the City and the City and the City and the City assembly of the City and the City assembly of the City 

(8.68884527683258,
 0.031914893617021274,
 2.5052826233294985,
 18.086780530074982,
 np.float64(6.731455784150346))

Without teacher forcing, the transformer really prefers using tokens relating to proper nouns (Ali-ahum, Aššur-malik, Ennam-Aššur),  places (City, assembly, colony), and money (mina, shekel). It shows an improvement compared to MBart trained on 5 epochs. It does get stuck on a loop of repetitive words. This behavior is supressed with teacher forcing. This makes sense, as the token predictions are being guided by the ground truth. 

#### Transformer w/ Teacher Forcing

In [22]:
test.test_model(model, test_loader, device, 4, "t", "raw", True)

METRICS
Test Loss     : 5.3302
Test Accuracy : 0.1702
BLEU          : 2.9824
chrF++        : 22.0422
Geometric Mean: 8.1079
PRED : From Isay to the City ers, the father er, and mina shekel and the City statithe ! and ed of the City and not ed. The 
REF  : Thus Kanesh, say to the payers, our messenger, every single colony, and the trading stations: A letter of the City has arrived. 
PRED : To the City of the City from to the --Reckoned the the the he from he with īin City assembly the he not pay he the City he -dhe City of the in colony add the If 
REF  : In the letter of the City it is written: From this day on, whoever buys meteoric iron, the City of Assur is not part of the profit made, the tithe on it Kanesh will collect. 
PRED : From to as to and been the father's and and to and been the the to the single I not been not to the not . but not staying the to the with not with the the to this ed that has to the the City ecto and the mina ce with the -a way of let City of the father's a

(5.330165863037109,
 0.1702127659574468,
 2.982365594802637,
 22.04224624011714,
 np.float64(8.107899655193886))

As expected, the repetitive token prediction occurs less often with teacher forcing. A lot more variety of different tokens are seen in the predictions. This is the best that our transformer architecture can do. Looking at the training epochs, the validation accuracy never went above 0.3087. This may be due to transformers needing lots of training data, and the number of entries that we have being insufficient for it.

### LSTM Testing

In [23]:
from contextlib import redirect_stdout
import io

with redirect_stdout(io.StringIO()):
    best, results = btester.evaluate_lstm("checkpoints/lstm", test_loader, device)

In [24]:
print(best)

lstm_seq2seq_epoch131_vAcc0.4305_vLoss3.9539.pth


In [25]:
import model.lstm as lstm

#Model epochs with good performance according to geomean
#lstm_seq2seq_epoch141_vAcc0.4322_vLoss3.9570.pth - 20.5095 (BEST)
#lstm_seq2seq_epoch143_vAcc0.4325_vLoss3.9505.pth - 16.6987
#lstm_seq2seq_epoch242_vAcc0.4380_vLoss3.9653.pth - 16.6698
#lstm_seq2seq_epoch118_vAcc0.4281_vLoss3.9584.pth - 15.4608
#lstm_seq2seq_epoch220_vAcc0.4380_vLoss3.9638.pth - 16.0713
#lstm_seq2seq_epoch184_vAcc0.4382_vLoss3.9586.pth - 16.0035
model = lstm.load_lstm_checkpoint("checkpoints/lstm/lstm_seq2seq_epoch141_vAcc0.4322_vLoss3.9570.pth", device)
test.test_model(model, test_loader, device, 4, "l", "raw", True)

Loaded epoch 141 | Val Acc: 0.4322 | Val Loss: 3.9570


METRICS
Test Loss     : 3.6209
Test Accuracy : 0.4149
BLEU          : 11.6626
chrF++        : 36.0675
Geometric Mean: 20.5095
PRED : From Kanesh, say to the messenger ers, our messenger er, and single colony and the trading stations: The letter of the colony and been ed. The 
REF  : Thus Kanesh, say to the payers, our messenger, every single colony, and the trading stations: A letter of the City has arrived. 
PRED : When the week of the City and is in engHe the day the after is ilthe iron the City and the to in delayof the City of de, then Kanesh on the and to go t. The 
REF  : In the letter of the City it is written: From this day on, whoever buys meteoric iron, the City of Assur is not part of the profit made, the tithe on it Kanesh will collect. 
PRED : The to as to have seized the case whoever to there to been in for to the verdict or two not it to the and s and the carries it and the and having in reit to the iron and will age then me attorney chanin of the shekel ce of the iron, 

(3.6209177374839783,
 0.4148936170212766,
 11.662556280002097,
 36.06746105002416,
 np.float64(20.509480597340595))

The LSTM has the best overall performance across all models tested, achieving a Geometric mean of 20.51 with teacher forcing, more than double the transformer's 8.11 and nearly three times than MBart's 7.23. This result is counterintuitive given that MBart has over 100 million pretrained parameters, but it makes sense given our constraints.

First, LSTM was trained for 250 epochs versus the transformer's effective early stop at epoch 114 and MBart's 5 epochs. The LSTM model has seen the training data far more times and is better calibrated to the specific vocabulary and sentence structures of the transliterations. The model architecture also helps out in the performance. The LSTM's recurrent state that saves long and short term memory is well-suited to remembering sentence structures, while a transformer's global attention is not that helpful when given lacking data. Third, unlike the transformer, the LSTM did not exhibit the severe repetition failure mode. It's sequential decoding means it naturally progresses through the output rather than repeatedly attending to the same source tokens.

The sample outputs confirm the LSTM captures the correct context and vocabulary: 
*"From Kanesh, say to the messengers, our messenger, every single colony and the trading stations: The letter of the colony and been ed."* 

This prediction may not be grammatically perfect but it captures the semantics very well. The model has correctly identified this as a message to a colony, it retrieved the right names and terms, and produced text that reads like Akkadian translations rather than random tokens.

In [26]:
# Best one without teacher forcing
model = lstm.load_lstm_checkpoint("checkpoints/lstm/lstm_seq2seq_epoch131_vAcc0.4305_vLoss3.9539.pth", device)
test.test_model(model, test_loader, device, 4, "l", "raw", False)

Loaded epoch 131 | Val Acc: 0.4305 | Val Loss: 3.9539


METRICS
Test Loss     : 8.1965
Test Accuracy : 0.0638
BLEU          : 7.5203
chrF++        : 23.3920
Geometric Mean: 13.2633
PRED : Thus karum Kanesh, say to the colony, and the trading stations, and the trading statione-girl and the colony has been sold for 
REF  : Thus Kanesh, say to the payers, our messenger, every single colony, and the trading stations: A letter of the City has arrived. 
PRED : When Iddin-abum will go to the City in the City and the attorney of the colony has been paid in the City and the sons of the colony has been paid in the City and the attorney of 
REF  : In the letter of the City it is written: From this day on, whoever buys meteoric iron, the City of Assur is not part of the profit made, the tithe on it Kanesh will collect. 
PRED : The day Aššur-šamšī answered: I submit to the City to the City assembly and the attorney of the City and the attorney of the City and the attorney of the City and the attorney of the City and the attorney of the City and the atto

(8.196497559547424,
 0.06382978723404255,
 7.520332274930926,
 23.3919996533815,
 np.float64(13.26331821108496))

Without teacher forcing the LSTM still substantially outperforms both the transformer and MBart, which reinforces that 250 epochs of training on this small, domain-specific corpus produces a better-calibrated model than either fewer training epochs (MBart) or an architecture that needs more data to converge (Transformer).

Analysis Ends Here.